# `mdpp` Example: Contact Analysis

This notebook demonstrates inter-residue contact analysis:

- `compute_contacts` -- per-frame inter-residue distances
- `compute_contact_frequency` -- contact frequency maps
- `compute_native_contacts` -- fraction of native contacts Q(t)

Use this notebook after the IO/alignment workflow.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.utils import auto_ticks

from mdpp.analysis.contacts import (
    compute_contact_frequency,
    compute_contacts,
    compute_native_contacts,
)
from mdpp.core.trajectory import align_trajectory, load_trajectory
from mdpp.plots.contacts import contact_frequency_to_matrix, plot_contact_map
from mdpp.plots.timeseries import plot_native_contacts

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)
traj = align_trajectory(traj, atom_selection="name CA")

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Inter-Residue Contacts


In [ ]:
contacts = compute_contacts(traj, scheme="closest-heavy")

print(f"Pairs: {contacts.residue_pairs.shape[0]}")
print(f"Mean distance: {contacts.distances_nm.mean():.3f} nm")

## Contact Frequency Map


In [ ]:
frequency, pairs = compute_contact_frequency(traj, cutoff_nm=0.45, scheme="closest-heavy")
n_residues = traj.topology.n_residues
matrix = contact_frequency_to_matrix(frequency, pairs, n_residues)

fig, ax = plt.subplots(figsize=(7, 6), dpi=120)
plot_contact_map(matrix, ax=ax)
ax.set_title("Contact Frequency Map (cutoff = 0.45 nm)")
auto_ticks(ax)
fig.tight_layout()

## Native Contacts Q(t)


In [ ]:
native = compute_native_contacts(traj, reference_frame=0, cutoff_nm=0.45, scheme="closest-heavy")

print(f"Native pairs: {native.native_pairs.shape[0]}")
print(f"Mean Q: {native.fraction.mean():.3f}")

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
plot_native_contacts(native, ax=ax, label="Q(t)")
ax.set_title("Fraction of Native Contacts")
auto_ticks(ax)
fig.tight_layout()